# Test Markdown Formatting

Quick test notebook for the markdown formatting feature using Claude 3 Haiku.

**Purpose:** Test the formatting step on a single markdown file without running the full DAG.

In [1]:
import sys
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

# Load environment variables
load_dotenv(find_dotenv(".env"), override=True)

# Add libs to path for imports
sys.path.insert(0, str(Path.cwd().parent / "libs"))

## Option A: Test with S3 (MinIO) File

Use this if you have markdown files in the `markdown-output` bucket.

In [2]:
from pdf_converter.s3_client import S3Client

# Initialize S3 client
s3_client = S3Client()

# List available markdown files
OUTPUT_BUCKET = "markdown-output"
md_files = s3_client.list_objects(OUTPUT_BUCKET, "")
md_files = [f for f in md_files if f.endswith(".md")]

print(f"Found {len(md_files)} markdown files in {OUTPUT_BUCKET}:")
for f in md_files[:10]:
    print(f"  - {f}")
if len(md_files) > 10:
    print(f"  ... and {len(md_files) - 10} more")

Found 70 markdown files in markdown-output:
  - ji-shi-liang-fang/pages_0011-0020.md
  - ji-shi-liang-fang/pages_0021-0030.md
  - ji-shi-liang-fang/pages_0031-0040.md
  - ji-shi-liang-fang/pages_0041-0050.md
  - ji-shi-liang-fang/pages_0051-0060.md
  - ji-shi-liang-fang/pages_0061-0070.md
  - ji-shi-liang-fang/pages_0071-0080.md
  - ji-shi-liang-fang/pages_0081-0090.md
  - ji-shi-liang-fang/pages_0091-0100.md
  - ji-shi-liang-fang/pages_0101-0110.md
  ... and 60 more


In [3]:
# Pick a file to format (change this to your file)
MARKDOWN_KEY = md_files[0] if md_files else "example.pdf/pages_0001-0010.md"
print(f"Selected file: {MARKDOWN_KEY}")

# Download original content
original_bytes = s3_client.download_bytes(OUTPUT_BUCKET, MARKDOWN_KEY)
original_markdown = original_bytes.decode("utf-8")

print(f"\nOriginal markdown ({len(original_markdown)} chars):")
print("=" * 60)
print(original_markdown[:2000])
if len(original_markdown) > 2000:
    print(f"\n... [{len(original_markdown) - 2000} more chars]")

Selected file: ji-shi-liang-fang/pages_0011-0020.md

Original markdown (3452 chars):
```markdown
# 头面部

## 头部

### 【肾水不足，邪火上冲，头似痛非痛】

**处方**: 熟地一两　玉竹一两　山萸肉四钱　真山药三钱　元参三钱　川芎三钱　当归三钱　五味子二钱　麦冬二钱

水煎服，重者三付断根。此方不可加减，若加减不应验，贻笑大方也。

### 【偏正头风顶上痛】

**辨症**: 属厥阴经头痛

**处方**: 川芎一钱　川羌活一钱　蒿本一钱　芸香二钱　细辛八分　薄荷三钱　银胡三钱　甘草一钱　水煎服。

**又方**: 细辛一钱　蔓荆子二钱　辛荑二钱　当归一两　川芎一两　水煎服，三付立愈。

**又方**: 用针灸法，左痛用针刺左迎香穴，右痛刺右迎香穴；左右皆痛，左右迎香穴同刺，立愈。

### 【偏正头风】

**处方**: 香白芷炒二两五钱　川芎炒　甘草炒　川乌头半生半熟各一两

研末，每服一钱，细茶、薄荷汤调下。百药不治，一服便可，天下第一方也。

### 【偏头痛或左右皆痛】

**辨症**: 少阳症头痛

**处方**: 川芎一两　白芍五钱　郁金一钱　柴胡一钱　香附二钱　芥子二钱　白芷五分　甘草一钱　水煎服，一付立愈。

**又方**: 白芷三钱　天麻一钱　防风一钱　荆芥五分　水煎服。

### 【一切头痛】

**处方**: 川芎四两　荆芥四两　白芷二两　甘草二钱　羌活二钱　防风钱半　细辛钱半

共细末，每服二钱，早晚清茶送下，准好。

**又方**: 川芎、白芷、石膏各等分共为细末，早晚清茶送下。

### 【男女一切头痛】

**处方**: 荆芥三钱　防风三钱　山栀子三钱　桔梗三钱　羌活三钱　川芎二钱　薄荷二钱　甘草二钱　胡连二钱　细辛钱半　水煎服。

### 【头痛】

**方名**: 血府逐瘀汤

**处方**: 当归三钱　桃仁四钱　红花三钱　枳壳二钱　赤芍二钱　柴胡一钱　甘草一钱　桔梗一钱半　川芎一钱半　牛膝二钱　生地黄四钱

**注**: 须查头痛患者无表症，无里症，无气症及痰饮等症；忽发忽好，百方不效，用此方一剂而愈。

**出处**: 王清任《医林改错》

### 【头风】

**处方**: 川芎一两　天麻一两　川乌一两（浸泡，刨

In [4]:
from pdf_converter.llm_client import LLMClient
from pdf_converter.markdown_formatter import FORMATTING_PROMPT, DEFAULT_FORMATTING_MODEL

# Initialize LLM client
llm_client = LLMClient()

print(f"Model: {DEFAULT_FORMATTING_MODEL}")
print(f"\nFormatting prompt:")
print(FORMATTING_PROMPT)

Model: anthropic/claude-3-haiku

Formatting prompt:
Clean up and fix the structure of this markdown document.

Tasks:
1. Fix heading hierarchy (ensure proper H1 → H2 → H3 nesting)
2. Remove artifacts (page numbers, headers/footers if duplicated)
3. Normalize formatting (consistent list styles, table alignment)
4. Remove excessive blank lines while preserving readability
5. Fix any broken tables or lists

Preserve all content - do not summarize or omit text.
Output only the formatted markdown, no explanations.


In [5]:
# Call the LLM to format the markdown
print("Calling Claude 3 Haiku for formatting...")

response = llm_client.call_text(
    model=DEFAULT_FORMATTING_MODEL,
    prompt=FORMATTING_PROMPT,
    content=original_markdown,
    max_tokens=16000,
)

formatted_markdown = response.content

print(f"\nFormatting complete!")
print(f"  Prompt tokens:     {response.prompt_tokens:,}")
print(f"  Completion tokens: {response.completion_tokens:,}")
print(f"  Cost:              ${response.cost_usd:.6f} USD")

Calling Claude 3 Haiku for formatting...

Formatting complete!
  Prompt tokens:     3,989
  Completion tokens: 3,856
  Cost:              $0.005817 USD


In [6]:
# Compare before and after
print("FORMATTED MARKDOWN:")
print("=" * 60)
print(formatted_markdown[:2000])
if len(formatted_markdown) > 2000:
    print(f"\n... [{len(formatted_markdown) - 2000} more chars]")

FORMATTED MARKDOWN:
# 头面部

## 头部

### 【肾水不足，邪火上冲，头似痛非痛】

**处方**: 熟地一两　玉竹一两　山萸肉四钱　真山药三钱　元参三钱　川芎三钱　当归三钱　五味子二钱　麦冬二钱

水煎服，重者三付断根。此方不可加减，若加减不应验，贻笑大方也。

### 【偏正头风顶上痛】

**辨症**: 属厥阴经头痛

**处方**: 川芎一钱　川羌活一钱　蒿本一钱　芸香二钱　细辛八分　薄荷三钱　银胡三钱　甘草一钱　水煎服。

**又方**: 细辛一钱　蔓荆子二钱　辛荑二钱　当归一两　川芎一两　水煎服，三付立愈。

**又方**: 用针灸法，左痛用针刺左迎香穴，右痛刺右迎香穴；左右皆痛，左右迎香穴同刺，立愈。

### 【偏正头风】

**处方**: 香白芷炒二两五钱　川芎炒　甘草炒　川乌头半生半熟各一两

研末，每服一钱，细茶、薄荷汤调下。百药不治，一服便可，天下第一方也。

### 【偏头痛或左右皆痛】

**辨症**: 少阳症头痛

**处方**: 川芎一两　白芍五钱　郁金一钱　柴胡一钱　香附二钱　芥子二钱　白芷五分　甘草一钱　水煎服，一付立愈。

**又方**: 白芷三钱　天麻一钱　防风一钱　荆芥五分　水煎服。

### 【一切头痛】

**处方**: 川芎四两　荆芥四两　白芷二两　甘草二钱　羌活二钱　防风钱半　细辛钱半

共细末，每服二钱，早晚清茶送下，准好。

**又方**: 川芎、白芷、石膏各等分共为细末，早晚清茶送下。

### 【男女一切头痛】

**处方**: 荆芥三钱　防风三钱　山栀子三钱　桔梗三钱　羌活三钱　川芎二钱　薄荷二钱　甘草二钱　胡连二钱　细辛钱半　水煎服。

### 【头痛】

**方名**: 血府逐瘀汤

**处方**: 当归三钱　桃仁四钱　红花三钱　枳壳二钱　赤芍二钱　柴胡一钱　甘草一钱　桔梗一钱半　川芎一钱半　牛膝二钱　生地黄四钱

**注**: 须查头痛患者无表症，无里症，无气症及痰饮等症；忽发忽好，百方不效，用此方一剂而愈。

**出处**: 王清任《医林改错》

### 【头风】

**处方**: 川芎一两　天麻一两　川乌一两（浸泡，刨去皮捣碎，炒黄）

以上药为细末，每服二钱，茶调下，薄荷更佳。

### 【秃疮良方】

**处方**: 苦陈皮（要细白皮）三两

瓦上炕干为细末，香油调

In [7]:
# Display formatted markdown rendered
from IPython.display import Markdown, display

print("Rendered formatted markdown:")
display(Markdown(formatted_markdown))

Rendered formatted markdown:


# 头面部

## 头部

### 【肾水不足，邪火上冲，头似痛非痛】

**处方**: 熟地一两　玉竹一两　山萸肉四钱　真山药三钱　元参三钱　川芎三钱　当归三钱　五味子二钱　麦冬二钱

水煎服，重者三付断根。此方不可加减，若加减不应验，贻笑大方也。

### 【偏正头风顶上痛】

**辨症**: 属厥阴经头痛

**处方**: 川芎一钱　川羌活一钱　蒿本一钱　芸香二钱　细辛八分　薄荷三钱　银胡三钱　甘草一钱　水煎服。

**又方**: 细辛一钱　蔓荆子二钱　辛荑二钱　当归一两　川芎一两　水煎服，三付立愈。

**又方**: 用针灸法，左痛用针刺左迎香穴，右痛刺右迎香穴；左右皆痛，左右迎香穴同刺，立愈。

### 【偏正头风】

**处方**: 香白芷炒二两五钱　川芎炒　甘草炒　川乌头半生半熟各一两

研末，每服一钱，细茶、薄荷汤调下。百药不治，一服便可，天下第一方也。

### 【偏头痛或左右皆痛】

**辨症**: 少阳症头痛

**处方**: 川芎一两　白芍五钱　郁金一钱　柴胡一钱　香附二钱　芥子二钱　白芷五分　甘草一钱　水煎服，一付立愈。

**又方**: 白芷三钱　天麻一钱　防风一钱　荆芥五分　水煎服。

### 【一切头痛】

**处方**: 川芎四两　荆芥四两　白芷二两　甘草二钱　羌活二钱　防风钱半　细辛钱半

共细末，每服二钱，早晚清茶送下，准好。

**又方**: 川芎、白芷、石膏各等分共为细末，早晚清茶送下。

### 【男女一切头痛】

**处方**: 荆芥三钱　防风三钱　山栀子三钱　桔梗三钱　羌活三钱　川芎二钱　薄荷二钱　甘草二钱　胡连二钱　细辛钱半　水煎服。

### 【头痛】

**方名**: 血府逐瘀汤

**处方**: 当归三钱　桃仁四钱　红花三钱　枳壳二钱　赤芍二钱　柴胡一钱　甘草一钱　桔梗一钱半　川芎一钱半　牛膝二钱　生地黄四钱

**注**: 须查头痛患者无表症，无里症，无气症及痰饮等症；忽发忽好，百方不效，用此方一剂而愈。

**出处**: 王清任《医林改错》

### 【头风】

**处方**: 川芎一两　天麻一两　川乌一两（浸泡，刨去皮捣碎，炒黄）

以上药为细末，每服二钱，茶调下，薄荷更佳。

### 【秃疮良方】

**处方**: 苦陈皮（要细白皮）三两

瓦上炕干为细末，香油调，搽数日，效①。

### 【偏头风方】

**处方**: 夏枯草四两　白芷三钱　芥穗三钱　苍耳子八钱　川芎二钱　细辛一钱　防风三钱　薄荷三钱　辛荑三钱　生黄芩三钱　柴胡三钱　生甘草三钱

### 【头发脱落】

**方名**: 通窍活血汤

**处方**: 麝香五厘绢包　桃仁三钱研泥　红花三钱　大枣七个去核　老葱三钱切碎　鲜姜三钱切碎　川芎二钱　赤芍一钱　黄酒半斤

煎三次，药汁倒在黄酒内，然后把麝香用绢包好，放在黄酒内再煎，熬六七沸，早晚空心服，喝三次。

**注**: 赤芍发散，白芍收敛，入肝经。此方亦治眼痛白珠红、鼻子臭、糟鼻子、年久耳聋、牙疳、出臭气、妇人乾劳、男子劳病、交节病作、小儿疳症。

**出处**: 王清任《医林改错》

### 【白发转黑】

**单方**: 轻粉十克，好醋为引，调匀，临睡时，涂在头发上用毛巾敷盖，早上取开全发变黑色。再加内服海螵蛸、何首乌各十五克，青黛十克，水煎服，三剂全愈。

### 【脱发】

**处方**: 生香油、桑叶，煎水去渣，洗头。

### 【少年白头】

**处方**: 黑芝麻、制首乌，做成小丸，每服六克。

### 【血热脱发】

**单方**: 熬大米稀饭快熟时放点熟黑芝麻，再加点冰糖，经常吃。

### 【头屑干洗方】

**处方**: 蒿本、白芷等分，为末，夜搽旦梳，垢自去也。另，桑白皮同柏叶，沐发不落。

### 【读书易记方】

**处方**: 远志、益智仁各十五克，桂圆肉三十克，研末，蜂蜜糖炼为丸，手指大。每次服三丸，每日二次。（一天比一天记忆增强）

**又方**: 蜂糖一两、鸡蛋白一个，调匀冲水服。每天早上一次，连服七天后，可以增强记忆力。

**出处**: 李光明传

### 【头晕眼花，手发胀，血压偏低，心内闷有郁气】

**处方**: 炙黄芪二十克　赤芍十二克　防风六克　山萸肉九克　柴胡十二克　羌活九克　山药九克　丹皮六克　熟地十二克（补肾水）川芎三克　当归十二克　生甘草四克（能泄火）

### 【经年头痛】

**单方**: 用家槐花晒干细末，煮熟鸡蛋蘸吃，喝黄酒发汗。

### 【血虚白发】

**处方**: 制首乌、熟地黄各十五克，水煎服。

**又方**: 制首乌十五克、生地黄三十克（酒洗），开水冲，代茶常饮。

## 面部

### 【紫印脸，青记脸黑如墨，白癜风，紫癜风】

**处方**: 见头部【头发脱落】条"通窍活血汤"

### 【紫白癜风】

**处方**: 贝母、南星等分为末，生姜带汁擦之。

**又方**: 用贝母、干姜等分为末，如澡豆，入密室中浴擦，得汗为妙。

### 【脸上眼下癍点】

**处方**: 白术二百克，用白米醋，浸泡七到十天，一天擦三次。

### 【口眼歪斜】

**方名**: 和血息火汤

**处方**: 升麻一钱　当归五钱　黄芪三钱　防风三钱　秦艽一钱　白芷五分　桂枝三分　天花粉二钱　甘草一钱　麦冬三钱　玄参五钱

煎服五剂见轻，九剂可愈。

**针法**: 用针刺颊车、地仓、百会、水沟、承浆，先泻后补。

## 眼眉部

### 【暴发火眼及云雾症】

**处方**: 硼砂三钱　枯矾二钱　胆矾三钱　明矾二钱　冰片二钱　炉甘石三钱　黄连三钱

共为细末，点眼即愈。

**又方**: 归尾　红花　胆矾　炉甘石各三钱

把药装在白布口袋内，用冷开水冲之，药色下即洗眼，三次立愈。

**又方**: 梅片、元寸、牛黄、硼砂、珠子、琥珀各等分，炉甘石三钱，共为细末，用纸卷药点眼，三四次即愈。①

### 【各种火眼】

**处方**: 明矾、胆矾、乌梅、川椒各三钱，雄鸡胆一个，新针七支，铜盆一个，开水三碗，把药同下盆内盖好，放热炕上，七天后，取出用纸过滤，将药水存入瓶内点眼。病情最重者，三五次即愈。

### 【眼伤，眼流水】

**处方**: 当归身四钱酒洗　白芍四钱　川芎三钱　生地四钱　杭芍三钱　花粉二钱半　防风二钱　丹皮三钱　青葙子二钱　枸杞子三钱（眼流水不用）竹叶为引，水煎服。

### 【前额连眉棱骨疼痛】

**辨症**: 阳明症头痛

**处方**: 防风　羌活　川芎　甘草各等分　水煎服。

### 【目赤肿痛】

**方名**: 息氛汤

**处方**: 白芍三钱　白蒺藜三钱　菊花三钱　山栀子三钱　当归三钱　茯苓三钱　柴胡二钱　花粉二钱　蔓荆子一钱　甘草一钱　决明子一钱

水煎服，服后洗三次愈。

### 【眼内云雾症，白内障】

**方名**: 太华山眼药膏（又名紫金锭）

**处方**: 炉甘石一斤　月石一斤　珊瑚三两　玛瑙三两　朱砂四两　石决明三两　绿豆粉四两　上片一两　溪片二两　炒片三两　元寸二钱

共细末　光明草①适量

熬膏为锭，赤金为衣，名退云散或紫金锭。

### 【各种眼疾】

**处方**: 黄连　黄柏　黄芩　蒙花　荆芥　防风各三钱　柴胡二钱　草决明三钱　木贼二钱　桑叶二钱　白芍花三钱　蚕沙一钱　甘草一钱　蝉蜕五分　浮风子三分　冰片一钱（后下）薄荷三钱半

用雪水泡药后，熬去渣，然后再下冰片。煎汤内服，也可收膏外用。

### 【眼中白翳，云雾胬肉】

**方名**: 天师膏

**处方**: 火硝一两　广丹一钱水飞　梅片五分

共细末点眼，云翳即退。

**注**: 此症现称白内障，白溢到最后眼珠长成一层黄白色，看不见。

**出处**: 张三丰祖师传

### 【风火烂眼良方】

**处方**: 川连二两　生炉甘石一两　端阳陈艾二两

用小磨香油调均，搽两夜，分块抹眼上，用之有效。

### 【眼生异物，白雾症】

**处方**: 青桐子花　酸枣仁　元明粉　羌活各一两为末

每服三钱连渣，日三服。

### 【眼痛，白珠红】

**方名**: 加味止痛没药散

**处方**: 没药三钱　血竭三钱　大黄三钱　朴硝二钱　石决明三钱

In [8]:
# Optional: Save formatted version back to S3 (uncomment to execute)
# print(f"Saving formatted version to s3://{OUTPUT_BUCKET}/{MARKDOWN_KEY}")
# s3_client.upload_text(formatted_markdown, OUTPUT_BUCKET, MARKDOWN_KEY)
# print("Done!")